# 1. pdf 페이지 단위로 청킹

In [21]:
'''
import os
import fitz  # PyMuPDF

def split_pdf_between_separators(
    filepath: str,
    output_dir: str = "split_pdf_sep",
    separator: str = "##########"
):
    """
    문서에서 separator 문자열의 위치를 모두 찾고,
    각 구간을 'separator 다음 ~ 다음 separator 직전' 으로 잘라 별도 PDF로 저장.
    - 페이지 중간이라도 클립해서 정확히 분리
    - 출력 파일명: 001.pdf, 002.pdf, ...
    """
    os.makedirs(output_dir, exist_ok=True)
    doc = fitz.open(filepath)

    # 1) 문서 전체에서 구분자의 (page_index, rect) 리스트 수집
    marks = []  # [(page_idx, rect), ...] rect: fitz.Rect
    for pno in range(len(doc)):
        page = doc[pno]
        rects = page.search_for(separator) or []
        # 같은 페이지에 여러 개 있을 수도 있으므로 모두 추가
        for r in rects:
            marks.append((pno, r))

    if len(marks) < 2:
        print("구분자가 2개 미만이라 분리할 구간이 없습니다.")
        doc.close()
        return []

    # 2) 문서 순서대로 정렬: 페이지번호, y좌표
    marks.sort(key=lambda pr: (pr[0], pr[1].y0))

    # 3) 인접한 구분자 쌍마다 하나의 구간 생성
    outputs = []
    for idx in range(len(marks) - 1):
        (start_page, start_rect) = marks[idx]
        (end_page, end_rect) = marks[idx + 1]

        # 새 문서
        out = fitz.open()

        # 각 페이지를 상황에 맞게 clip해서 out에 붙인다
        for pno in range(start_page, end_page + 1):
            src_page = doc[pno]
            page_rect = src_page.rect

            # 기본은 전체 페이지
            clip = page_rect

            if start_page == end_page:
                # 같은 페이지 안에서 'start_rect 아래' ~ 'end_rect 위' 영역만
                clip = fitz.Rect(page_rect.x0, start_rect.y1, page_rect.x1, end_rect.y0)
            elif pno == start_page:
                # 시작 페이지: 구분자 '아래쪽'만 포함
                clip = fitz.Rect(page_rect.x0, start_rect.y1, page_rect.x1, page_rect.y1)
            elif pno == end_page:
                # 끝 페이지: 구분자 '위쪽'만 포함
                clip = fitz.Rect(page_rect.x0, page_rect.y0, page_rect.x1, end_rect.y0)
            else:
                # 중간 페이지: 전체
                clip = page_rect

            # clip 높이가 유효할 때만 추가
            if clip.height > 1 and clip.width > 1:
                # 새 페이지 만들고, 소스 PDF의 해당 페이지를 clip하여 채움
                new_page = out.new_page(width=page_rect.width, height=page_rect.height)
                # 소스 PDF의 pno 페이지를 new_page에 그대로(벡터 유지) 표시
                new_page.show_pdf_page(new_page.rect, doc, pno, clip=clip)

        # 빈 문서가 아니면 저장
        if out.page_count > 0:
            out_path = os.path.join(output_dir, f"{idx+1:03d}.pdf")
            out.save(out_path)
            out.close()
            outputs.append(out_path)
            print(f"저장: {out_path}  (from mark {idx} to {idx+1})")

    doc.close()
    return outputs


# ===== 사용 예시 =====
if __name__ == "__main__":
    filepath = "해외조직망정산지침_최종_251018.pdf"  # 원본 파일명
    result_files = split_pdf_between_separators(
        filepath,
        output_dir="split_pdf_sep",
        separator="##########"
    )
    print("총 분리된 파일 수:", len(result_files))
'''


저장: split_pdf_sep/001.pdf  (from mark 0 to 1)
저장: split_pdf_sep/002.pdf  (from mark 1 to 2)
저장: split_pdf_sep/003.pdf  (from mark 2 to 3)
저장: split_pdf_sep/004.pdf  (from mark 3 to 4)
저장: split_pdf_sep/005.pdf  (from mark 4 to 5)
저장: split_pdf_sep/006.pdf  (from mark 5 to 6)
저장: split_pdf_sep/007.pdf  (from mark 6 to 7)
저장: split_pdf_sep/008.pdf  (from mark 7 to 8)
저장: split_pdf_sep/009.pdf  (from mark 8 to 9)
저장: split_pdf_sep/010.pdf  (from mark 9 to 10)
저장: split_pdf_sep/011.pdf  (from mark 10 to 11)
저장: split_pdf_sep/012.pdf  (from mark 11 to 12)
저장: split_pdf_sep/013.pdf  (from mark 12 to 13)
저장: split_pdf_sep/014.pdf  (from mark 13 to 14)
저장: split_pdf_sep/015.pdf  (from mark 14 to 15)
저장: split_pdf_sep/016.pdf  (from mark 15 to 16)
저장: split_pdf_sep/017.pdf  (from mark 16 to 17)
저장: split_pdf_sep/018.pdf  (from mark 17 to 18)
저장: split_pdf_sep/019.pdf  (from mark 18 to 19)
저장: split_pdf_sep/020.pdf  (from mark 19 to 20)
저장: split_pdf_sep/021.pdf  (from mark 20 to 21)
저장: split_p

# 2. 페이지별로 document parser 적용 (markdown 변환)
- PyMuPDF + Upstage 변환

In [1]:
# pip install -qU langchain-core langchain-upstage markdownify

import os
from langchain_upstage import UpstageDocumentParseLoader
from markdownify import markdownify as md

os.environ["UPSTAGE_API_KEY"] = "up_UmN0IgXpQ8eOzwrt2kN386ctcla7C"

### 2-1. hwp 바로 읽기 (성공)

In [2]:
#pip install -qU langchain-teddynote

from langchain_teddynote.document_loaders import HWPLoader

# HWP Loader 객체 생성
loader = HWPLoader("해외조직망정산지침_최종_251118.hwp")

# 문서 로드
docs = loader.load()
# 결과 출력
print(docs[0].page_content[:1000])

해외조직망 정산지침K O T R A목    차1조. 목 적 2조. 적용범위 3조. 회계담당자 1. 정의  2. 회계처리에 대한 법적 책임 3. 자금관리에 대한 책임 4. 회계담당자의 임명4조. 회계장부 1. 장부의 종류 2. 장부작성 5조. 자금관리 및 집행   1. 계좌개설   2. 자금차입   3. 자금집행   4. 공용카드 사용   5. 인터넷 뱅킹 또는 자금이체를 통한 자금집행 6. 현금 집행 7. 환차손익 처리   8. 수입 및 환입처리   9. 자산관리  10. 계약체결(용역계약 포함) 6조. 정산업무 개요 및 정산서 작성   1. 정산업무 흐름 개요  2. 주요 정산개념 및 정산양식 작성 요령  3. 소유지불 처리  4. 분리정산7조. 정산보고   1. 분기말 정산보고  2. 연도말 정산보고   3. 특정사업 정산보고  4. 조직망 폐쇄시 정산보고  5. 정산보고 시 유의사항 8조. 비목별 세부 정산요령   1. 인건비   2. 조직망운영비  3. 임차료  5. 공기구비품비 및 차량운반구  6. 교육훈련비   7. 보증금(예치금 포함)   8. 단기선급비 및 장기선급비  9. 장치비  10. 장기성 유동부채   11. 지급이자와 할인료  12. 본사 주요사업단위 지원자금  13. 행사비  14. 소액경비 및 다발성 경비 9조. 정산 시 유의사항 (공통사항) 1. 영수증 2. 수표 3. 정산 시기  4. 지적사항에 대한 조치 5. 관련 법규 및 지침 준수  6. 기 타10조. 정산에 따른 제재조치 및 감점제도 운영  1. 근거  2. 제재내용   3. 업적평가 행정감점제도 운영 ##########1조.  목  적 1. 본 지침은 공사 회계기준시행세칙 제6장 66조(세부지침)가 정하는 바에 의하여 해외무역관, 수출인큐베이터, 해외IT지원센터, 박람회 및 전시회 한국관(이하 “해외조직망”이라 칭함.) 파견요원에게 회계실무처리에 대한 절차와 방법을 습득하게 하여 효율적인 회계업무처리를 하게 함을 목적으로 한다.2조.  적용범위 1. 해외조직망의 회계업무처

In [3]:
import re
from typing import List, Tuple
from langchain.schema import Document

def extract_page_num_simple(text: str) -> str:
    """
    page_content의 첫 글자부터 '다음 숫자(.)' 직전까지 반환.
    공백이 없어도(예: '회계장부1.') 동작하도록 패턴 보정.
    '2.' 뿐 아니라 '2.1.' 같은 번호도 감지.
    """
    s = text.strip()
    s = re.sub(r'\s+', ' ', s)  # 보기 좋은 공백 정규화(선택)

    # 핵심 변경: \s* 로 '숫자 앞 공백'을 선택적으로, '소수점 번호'도 허용
    m = re.match(r'^(.*?)(?=\s*\d+(?:\.\d+)*\.\s*|$)', s)
    return m.group(1).strip() if m else s

def split_docs_by_global_separators(
    docs: List[Document],
    origin_pdf: str,
    separator: str = "##########",
    keep_empty: bool = False,
) -> List[Document]:
    sep_pat = re.escape(separator)
    marks: List[Tuple[int, int, int]] = []

    for di, d in enumerate(docs):
        text = d.page_content or ""
        for m in re.finditer(sep_pat, text):
            marks.append((di, m.start(), m.end()))

    if len(marks) < 2:
        return []

    marks.sort(key=lambda t: (t[0], t[1]))

    def extract_span_text(docs, start_pos, end_pos) -> str:
        (s_di, s_off) = start_pos
        (e_di, e_off) = end_pos
        if (s_di > e_di) or (s_di == e_di and s_off >= e_off):
            return ""

        parts = []
        st = docs[s_di].page_content or ""
        parts.append(st[s_off: (len(st) if s_di != e_di else e_off)])
        for di in range(s_di + 1, e_di):
            parts.append(docs[di].page_content or "")
        if e_di != s_di:
            et = docs[e_di].page_content or ""
            parts.append(et[:e_off])
        return "".join(parts)

    out_docs: List[Document] = []

    for idx in range(len(marks) - 1):
        prev_doc, _, prev_end = marks[idx]
        next_doc, next_start, _ = marks[idx + 1]
        span_text = extract_span_text(
            docs, start_pos=(prev_doc, prev_end), end_pos=(next_doc, next_start)
        ).strip()

        if not span_text and not keep_empty:
            continue

        page_num = extract_page_num_simple(span_text)

        out_docs.append(
            Document(
                page_content=span_text,
                metadata={
                    "origin_pdf": origin_pdf,
                    "page_num": page_num,
                },
            )
        )

    return out_docs

In [4]:
# ===== 사용 예시 =====
split_docs = split_docs_by_global_separators(
    docs,
    origin_pdf="해외조직망정산지침",
    separator="##########",
)
print(split_docs[0].metadata)
print(split_docs[0].page_content[:200])

{'origin_pdf': '해외조직망정산지침', 'page_num': '1조. 목 적'}
1조.  목  적 1. 본 지침은 공사 회계기준시행세칙 제6장 66조(세부지침)가 정하는 바에 의하여 해외무역관, 수출인큐베이터, 해외IT지원센터, 박람회 및 전시회 한국관(이하 “해외조직망”이라 칭함.) 파견요원에게 회계실무처리에 대한 절차와 방법을 습득하게 하여 효율적인 회계업무처리를 하게 함을 목적으로 한다.2조.  적용범위 1. 해외조직망의 회계업무


In [7]:
#print(split_docs[0].page_content)
len(split_docs)

34

### 2-2. pdf > html > md (실패)

#### - 변환은 잘 되는데 계층 구조 인식이 잘 안됨

In [36]:
'''
# 2️⃣ PDF 경로
file_path = "026.pdf"
md_path = "026.md"

# 3️⃣ PDF → HTML (Upstage)
loader = UpstageDocumentParseLoader(file_path, ocr="force") 
pages = loader.load()

# 4️⃣ HTML → Markdown 변환
md_pages = []
for page in pages:
    html_content = page.page_content
    markdown_text = md(html_content, heading_style="ATX")
    # 🚫 Page 1, Page 2 제거. 구분선만 유지
    md_pages.append(f"\n\n---\n\n{markdown_text}")

# 5️⃣ 파일로 저장
with open(md_path, "w", encoding="utf-8") as f:
    f.write("".join(md_pages).lstrip())

print(f"✅ Markdown 파일 저장 완료: {os.path.abspath(md_path)}")
'''

✅ Markdown 파일 저장 완료: /Users/dayeayim/streamlit-finance-new/026_false.md


#### - 계층구조 임의지정 (실패)

In [42]:
'''
import os
import re
from pathlib import Path
from bs4 import BeautifulSoup
from markdownify import markdownify as md
from langchain_upstage import UpstageDocumentParseLoader

# ====== 패턴(규칙) 강화 ======
RE_ARTICLE = re.compile(r"^\s*\d+조[.\s]")                       # # 8조.
RE_NUM     = re.compile(r"^\s*\d+[.)]\s*")                       # ## 5. / 5)
RE_KOR     = re.compile(r"^\s*[가-하][.)]\s*")                    # ### 가. / 나)
RE_NUMSUB  = re.compile(r"^\s*(?:\d+[)]|\(\d+\)|[①-⑳])\s*")      # #### 1) / (1) / ①
RE_KORSUB  = re.compile(r"^\s*[가-하][)]\s*$")                    # ##### 가) 단독행
RE_BULLET   = re.compile(r"^\s*[-•·–]\s+")                       # -, •, ·, en dash
RE_BULLET_O = re.compile(r"^\s*ㅇ\s+")

# 문장 내부 마커 경계 (공백 없어도 동작)
SPLIT_MARKERS = re.compile(
    r"(?=(?:\d+[).]\s*|\(\d+\)\s*|[①-⑳]\s*|[가-하][).]\s*))"
)

def clean_text(s: str) -> str:
    if not s: return s
    # NBSP/제로폭 제거
    s = s.replace("\xa0", " ").replace("\u200b", "").replace("\u200c", "").replace("\u200d", "")
    # 중복 공백 정리
    s = re.sub(r"[ \t]{2,}", " ", s)
    return s.strip()

def split_by_markers_or_breaks(text: str):
    """개행으로 1차, 이어붙은 마커(1)·가) 등)로 2차 분리"""
    text = clean_text(text)
    raw_lines = [x for x in re.split(r"\n+", text) if x.strip()]
    out = []
    for ln in raw_lines:
        parts = [p.strip() for p in SPLIT_MARKERS.split(ln) if p.strip()]
        out.extend(parts if parts else [ln.strip()])
    return [x for x in out if x]

def split_lines_from_p(p_tag):
    """
    <p> 내부를 안전하게 평탄화 후 마커로 분리
    - separator=' ' 로 인라인 엘리먼트 사이 공백 보존
    - <br>가 있어도 get_text가 처리하므로 별도 처리 불필요
    """
    text = p_tag.get_text(separator=" ")
    return split_by_markers_or_breaks(text)

def classify_line(line: str):
    s = clean_text(line)
    if not s:
        return (None, 'empty', s)

    # 헤딩: 접두 보존(표시 그대로)
    if RE_ARTICLE.match(s): return (1, 'heading', s)  # # 8조.
    if RE_NUM.match(s):     return (2, 'heading', s)  # ## 5. / 5)
    if RE_KOR.match(s):     return (3, 'heading', s)  # ### 가. / 나)
    if RE_NUMSUB.match(s):  return (4, 'heading', s)  # #### 1) / (1) / ①
    if RE_KORSUB.match(s):  return (5, 'heading', s)  # ##### 가) 단독

    # 불릿
    if RE_BULLET.match(s):   return (None, 'ul', RE_BULLET.sub("", s, 1).strip())
    if RE_BULLET_O.match(s): return (None, 'ul', RE_BULLET_O.sub("", s, 1).strip())

    return (None, 'p', s)

def normalize_html_structure(upstage_html: str) -> str:
    """
    - 마커(조/숫자/한글/원형숫자) 기반으로 줄 분리
    - 헤딩 레벨 매핑: #조. → H1, 숫자 → H2, 가. → H3, 1) → H4, 가)단독 → H5
    - 불릿(-, •, ·, ㅇ, –) → <ul><li>
    - <hr> 제거
    """
    soup = BeautifulSoup(upstage_html, "html.parser")

    # 수평선 제거
    for hr in soup.find_all("hr"): hr.decompose()

    # 1) 이미 헤딩처럼 보이는 태그는 레벨 교정
    for tag in list(soup.find_all(['h1','h2','h3','h4','h5','p'])):
        text = clean_text(tag.get_text(" ", strip=True))
        lvl, kind, content = classify_line(text)
        if kind == 'heading':
            tag.name = {1:'h1', 2:'h2', 3:'h3', 4:'h4', 5:'h5'}.get(lvl, 'h3')
            tag.string = content

    # 2) 일반 문단을 마커로 분리해 헤딩/리스트로 재구성
    candidates = list(soup.find_all("p"))
    for p in candidates:
        lines = split_lines_from_p(p)
        if not lines or len(lines) == 1 and lines[0] == clean_text(p.get_text()):
            # 분리 필요 없으면 스킵
            continue

        block = soup.new_tag("div")
        current_ul = None
        for ln in lines:
            lvl, kind, content = classify_line(ln)

            if kind == 'heading':
                current_ul = None
                htag = {1:'h1', 2:'h2', 3:'h3', 4:'h4', 5:'h5'}.get(lvl, 'h3')
                h = soup.new_tag(htag); h.string = content
                block.append(h)

            elif kind == 'ul':
                if current_ul is None:
                    current_ul = soup.new_tag("ul")
                    block.append(current_ul)
                li = soup.new_tag("li"); li.string = content
                current_ul.append(li)

            else:
                current_ul = None
                para = soup.new_tag("p"); para.string = content
                block.append(para)

        p.replace_with(block)

    # 마지막 안전장치
    for hr in soup.find_all("hr"): hr.decompose()
    return str(soup)

# ============== 실행부 ==============

file_path = "026.pdf"
md_path   = "026_tag2.md"

loader = UpstageDocumentParseLoader(file_path, ocr="force")   # 요청대로 force 유지
pages = loader.load()

md_parts = []
for page in pages:
    html_content = page.page_content
    normalized_html = normalize_html_structure(html_content)
    md_text = md(
        normalized_html,
        heading_style="ATX",
        strip=["style","script"],
        bullets="-"
    ).strip()
    md_parts.append(md_text)

md_final = "\n\n".join(md_parts)
# 혹시 남아있을 수 있는 단독 '---' 제거
md_final = re.sub(r"(?m)^\s*-{3,}\s*$", "", md_final).strip()

with open(md_path, "w", encoding="utf-8") as f:
    f.write(md_final)

print(f"✅ Markdown 파일 저장 완료: {os.path.abspath(md_path)}")
'''


✅ Markdown 파일 저장 완료: /Users/dayeayim/streamlit-finance-new/026_tag2.md
